# Notebook 01 — STAC Query

**What you will learn:**
- What the SpatioTemporal Asset Catalog (STAC) specification is
- How to search a public STAC catalog with `pystac-client`
- What a STAC Item contains and how its assets relate to actual raster data

---

## Background: What is STAC?

STAC is a standard for describing geospatial datasets so that they can be discovered and queried consistently across providers. It has three levels:

| Level | What it is |
|-------|------------|
| **Catalog** | The root — a searchable index of collections |
| **Collection** | A named group of related items (e.g. all Sentinel-2 L2A scenes) |
| **Item** | One scene: metadata (date, cloud cover, bbox) + links to the actual files |
| **Asset** | A file linked from an Item — e.g. a COG GeoTIFF for one band |

The actual raster data lives in the **Assets** — not in the catalog itself. STAC is purely a metadata and discovery layer.

We will query the **Element84 earth-search** catalog, which indexes Sentinel-2 L2A scenes hosted on AWS S3 as Cloud-Optimized GeoTIFFs (COGs). No authentication is required.

In [1]:
import pystac_client
import pystac
import sys
sys.path.insert(0, "..")
from utils.stac_helpers import search_sentinel2, inspect_item

CATALOG_URL = "https://earth-search.aws.element84.com/v1"

# Open the catalog — this is just an HTTP request to fetch the root JSON
catalog = pystac_client.Client.open(CATALOG_URL)
print("Catalog title:", catalog.title)
print("Collections available:")
for col in catalog.get_collections():
    print(" ", col.id)

Catalog title: Earth Search by Element 84
Collections available:
  sentinel-2-pre-c1-l2a
  cop-dem-glo-30
  naip
  cop-dem-glo-90
  landsat-c2-l2
  sentinel-2-l2a
  sentinel-2-l1c
  sentinel-2-c1-l2a
  sentinel-1-grd


## Search for Sentinel-2 scenes

We search by:
- **bbox**: a bounding box in the Southeastern PA
- **datetime**: June–August 2023 (low cloud cover, growing season)
- **cloud cover**: less than 20%

The search returns a lazy iterator — nothing is downloaded until we call `list()`.

In [2]:
BBOX = [-75.75, 39.5, -75.25, 40.0]   # [lon_min, lat_min, lon_max, lat_max]
DATE_RANGE = "2023-06-01/2023-08-31"

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=10,
)

items = list(search.items())
print(f"Found {len(items)} scenes")
for item in items:
    print(f"  {item.datetime.date()}  cloud={item.properties.get('eo:cloud_cover'):.1f}%  id={item.id}")

Found 3 scenes
  2023-08-13  cloud=12.7%  id=S2A_18SVJ_20230813_0_L2A
  2023-06-04  cloud=18.7%  id=S2A_18SVJ_20230604_0_L2A
  2023-06-04  cloud=10.0%  id=S2A_18TVK_20230604_0_L2A


## Inspect a single STAC Item

A STAC Item is a GeoJSON Feature with extra fields. The key parts are:
- `.id` — unique scene identifier
- `.datetime` — acquisition timestamp
- `.bbox` — bounding box in WGS84
- `.properties` — metadata dict (cloud cover, platform, etc.)
- `.assets` — dict of named file links

In [3]:
item = items[0]

print("=== Item overview ===")
inspect_item(item)

print("\n=== Full properties dict ===")
for k, v in item.properties.items():
    print(f"  {k}: {v}")

=== Item overview ===
ID:         S2A_18SVJ_20230813_0_L2A
Date:       2023-08-13 16:02:25.315000+00:00
Cloud cover:12.651107%
BBox:       [-76.16761290616306, 38.755202928221415, -74.88608605736992, 39.7502030226425]
Assets:     ['aot', 'blue', 'coastal', 'granule_metadata', 'green', 'nir', 'nir08', 'nir09', 'red', 'rededge1', 'rededge2', 'rededge3', 'scl', 'swir16', 'swir22', 'thumbnail', 'tileinfo_metadata', 'visual', 'wvp', 'aot-jp2', 'blue-jp2', 'coastal-jp2', 'green-jp2', 'nir-jp2', 'nir08-jp2', 'nir09-jp2', 'red-jp2', 'rededge1-jp2', 'rededge2-jp2', 'rededge3-jp2', 'scl-jp2', 'swir16-jp2', 'swir22-jp2', 'visual-jp2', 'wvp-jp2']

Sample asset href (red band):
  https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/2023/8/S2A_18SVJ_20230813_0_L2A/B04.tif

=== Full properties dict ===
  created: 2023-08-14T04:21:56.502Z
  platform: sentinel-2a
  constellation: sentinel-2
  instruments: ['msi']
  eo:cloud_cover: 12.651107
  mgrs:utm_zone: 18
  mgrs:latitude_b

## Examine the Assets

Each asset is a named link to a file. For Sentinel-2 L2A on earth-search, the band assets are named by colour/role (not by Sentinel band number):

| Asset key | Sentinel band | Wavelength |
|-----------|--------------|------------|
| `blue`    | B02          | 490 nm     |
| `green`   | B03          | 560 nm     |
| `red`     | B04          | 665 nm     |
| `nir`     | B08          | 842 nm     |
| `scl`     | —            | Scene Classification Layer |

Each href points to a **Cloud-Optimized GeoTIFF** (COG) on AWS S3. You can read a spatial subset of it without downloading the whole file — that is exactly what xarray + odc-stac will do in notebook 02.

In [4]:
print("=== All assets for this scene ===")
for name, asset in item.assets.items():
    print(f"  {name:20s}  {asset.media_type or '':40s}  {asset.href[:80]}")

print("\n=== Red band asset in detail ===")
red_asset = item.assets["red"]
print("href:       ", red_asset.href)
print("media_type: ", red_asset.media_type)
print("roles:      ", red_asset.roles)

# The href is a COG on S3. The URL structure is:
#   s3://sentinel-cogs/sentinel-s2-l2a-cogs/<tile>/<year>/<month>/<scene_id>/<band>.tif
# odc-stac will use HTTP range requests against this URL to read only our bbox.

=== All assets for this scene ===
  aot                   image/tiff; application=geotiff; profile=cloud-optimized  https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/20
  blue                  image/tiff; application=geotiff; profile=cloud-optimized  https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/20
  coastal               image/tiff; application=geotiff; profile=cloud-optimized  https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/20
  granule_metadata      application/xml                           https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/20
  green                 image/tiff; application=geotiff; profile=cloud-optimized  https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/20
  nir                   image/tiff; application=geotiff; profile=cloud-optimized  https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/18/S/VJ/20
  ni

## Use the helper function

`utils/stac_helpers.py` wraps the search logic so later notebooks can reuse it without repeating setup code.

In [5]:
# Same search, via the helper
items_via_helper = search_sentinel2()
print(f"Helper returned {len(items_via_helper)} items")
assert len(items_via_helper) == len(items), "Helper should return same count"

Helper returned 3 items


## Validation

In [6]:
assert len(items) >= 2, "Expected at least 2 scenes in the date range"
assert "red" in items[0].assets, "Missing red band asset"
assert "nir" in items[0].assets, "Missing nir band asset"
assert items[0].bbox is not None, "Item should have a bounding box"
assert "eo:cloud_cover" in items[0].properties, "Missing cloud cover property"
print("All assertions passed.")

All assertions passed.


---
## Learning Checkpoint — Q&A

Answer these before moving to notebook 02. They should be answerable from what you observed above.

**Q1:** What is a STAC Item, and how does it differ from the actual raster data?

> *A STAC Item is a metadata record (GeoJSON Feature) describing one scene — its location, date, cloud cover, and links to files. The actual pixel data lives in the Assets (COG files on S3), not in the Item itself. The Item is a discovery object; the Asset is the data.*

**Q2:** What does `eo:cloud_cover` represent, and where does it live in the STAC data model?

> *It is the estimated percentage of the scene obscured by cloud. It lives in `item.properties` — the metadata dict attached to every Item. The `eo:` prefix means it comes from the Electro-Optical STAC extension.*

**Q3:** Why is the asset href a URL pointing to a COG rather than raw data?

> *A COG (Cloud-Optimized GeoTIFF) stores its data in a tiled, internally overviewed layout so that HTTP range requests can fetch only the bytes covering a specific spatial window — without downloading the entire file. This makes it efficient to read a small bbox from a scene that covers hundreds of km².*